In [10]:
import pandas as pd
import numpy as np
import os
import math

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F


from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from scipy.stats import pearsonr

import geopandas as gpd
from shapely.geometry import Point
from geopandas import GeoDataFrame
from geopy.distance import geodesic
from datetime import date, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectFromModel

import wrangling as wr
from einops.layers.torch import Rearrange
from torch.nn import LayerNorm
from scipy import stats

In [15]:
def roll_feature(X):
    N, feature_num, time = X.shape
    X_transform = None
    for i in range(N):
        X_tr = pd.DataFrame(index=range(feature_num), dtype=np.float64)
        for j in range(feature_num):
            x = X[i,j,:]
            X_tr.loc[j, 'ave'] = x.mean()
            X_tr.loc[j, 'std'] = x.std()
            X_tr.loc[j, 'max'] = x.max()
            X_tr.loc[j, 'min'] = x.min()
            X_tr.loc[j, 'q01'] = np.quantile(x, 0.01)
            X_tr.loc[j, 'q05'] = np.quantile(x, 0.05)
            X_tr.loc[j, 'q95'] = np.quantile(x, 0.05)
            X_tr.loc[j, 'q99'] = np.quantile(x, 0.95)
            X_tr.loc[j, 'abs_median'] = np.median(np.abs(x))
            X_tr.loc[j, 'abs_q95'] = np.quantile(np.abs(x),0.95)
            X_tr.loc[j, 'abs_q99'] = np.quantile(np.abs(x),0.99)
            X_tr.loc[j, 'abs_median'] = np.median(np.abs(x))
            X_tr.loc[j, 'F_test'], X_tr.loc[j, 'p_test'] = stats.f_oneway(x[:30], x[30:60], x[60:90], x[90:120], x[120:150], x[150:180], x[180:])
            X_tr.loc[j, 'av_change_abs'] = np.mean(np.diff(x))
            # X_tr.loc[j, 'av_change_rate'] = np.mean(np.nonzero((np.diff(x) / x[:-1]))[0])
            X_tr.loc[j, 'abs_max'] = np.abs(x).max()

            for windows in [10, 30]:
                x_series = pd.Series(x)
                x_roll_std = x_series.rolling(windows).std().dropna().values
                x_roll_mean = x_series.rolling(windows).mean().dropna().values
                segment = j

                X_tr.loc[segment, 'ave_roll_std_' + str(windows)] = x_roll_std.mean()
                X_tr.loc[segment, 'std_roll_std_' + str(windows)] = x_roll_std.std()
                X_tr.loc[segment, 'max_roll_std_' + str(windows)] = x_roll_std.max()
                X_tr.loc[segment, 'min_roll_std_' + str(windows)] = x_roll_std.min()
                X_tr.loc[segment, 'q01_roll_std_' + str(windows)] = np.quantile(x_roll_std, 0.01)
                X_tr.loc[segment, 'q05_roll_std_' + str(windows)] = np.quantile(x_roll_std, 0.05)
                X_tr.loc[segment, 'q95_roll_std_' + str(windows)] = np.quantile(x_roll_std, 0.95)
                X_tr.loc[segment, 'q99_roll_std_' + str(windows)] = np.quantile(x_roll_std, 0.99)
                X_tr.loc[segment, 'av_change_abs_roll_std_' + str(windows)] = np.mean(np.diff(x_roll_std))
                # X_tr.loc[segment, 'av_change_rate_roll_std_' + str(windows)] = np.mean(
                #     np.nonzero((np.diff(x_roll_std) / x_roll_std[:-1]))[0])
                X_tr.loc[segment, 'abs_max_roll_std_' + str(windows)] = np.abs(x_roll_std).max()

                X_tr.loc[segment, 'ave_roll_mean_' + str(windows)] = x_roll_mean.mean()
                X_tr.loc[segment, 'std_roll_mean_' + str(windows)] = x_roll_mean.std()
                X_tr.loc[segment, 'max_roll_mean_' + str(windows)] = x_roll_mean.max()
                X_tr.loc[segment, 'min_roll_mean_' + str(windows)] = x_roll_mean.min()
                X_tr.loc[segment, 'q01_roll_mean_' + str(windows)] = np.quantile(x_roll_mean, 0.01)
                X_tr.loc[segment, 'q05_roll_mean_' + str(windows)] = np.quantile(x_roll_mean, 0.05)
                X_tr.loc[segment, 'q95_roll_mean_' + str(windows)] = np.quantile(x_roll_mean, 0.95)
                X_tr.loc[segment, 'q99_roll_mean_' + str(windows)] = np.quantile(x_roll_mean, 0.99)
                X_tr.loc[segment, 'av_change_abs_roll_mean_' + str(windows)] = np.mean(np.diff(x_roll_mean))
                # X_tr.loc[segment, 'av_change_rate_roll_mean_' + str(windows)] = np.mean(
                #     np.nonzero((np.diff(x_roll_mean) / x_roll_mean[:-1]))[0])
                X_tr.loc[segment, 'abs_max_roll_mean_' + str(windows)] = np.abs(x_roll_mean).max()

        shape = X_tr.shape
        X_tr = np.array(X_tr)
        X_tr = X_tr.reshape([1] + list(shape))
        if X_transform is None:
            X_transform = X_tr
        else:
            X_transform = np.concatenate((X_transform, X_tr))

        if i%1000 ==0:
            print(X_transform.shape)
            print('Finished {} records in rolling'.format(i))

    print(X_transform.shape)
    # X_transform.dump('../data/X_roll_feature')
    return X_transform

In [16]:
####### Load data

y_df = pd.read_csv('/Users/guptsh/Downloads/Soil_Land_Crop/datasets/gridMET_data/modified/label.csv')
y_df['pred'] = np.nan

contents = ['tmmn', 'tmmx', 'pr', 'srad', 'sph', 'rmin', 'rmax', 'vs', 'th',  # 9 main
            'vpd', 'pet', 'etr', 'erc', 'bi', 'fm100', 'fm1000']  # except pdsi so far
X = wr.read_feature_data(contents)
X, wind_zero_idx = wr.feature_engineering(X, contents, norm = "std", intersect = False)

pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
pre-shape (45338, 210)
post-shape (1, 45338, 210)
vs idx is 7
(45338, 19, 210)


In [ ]:
X_rolled = roll_feature(X)

(1, 19, 55)
Finished 0 records in rolling
(1001, 19, 55)
Finished 1000 records in rolling
